In [2]:
%%writefile app.py

# 파이썬 기본 경고 메시지(DeprecationWarning, UserWarning 등)가 콘솔/화면에 출력되지 않도록 숨김 처리합니다.
import warnings

warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# 운영체제 환경변수 접근, 정규표현식 처리, URL 인코딩 및 HTTP 요청 처리를 위한 파이썬 표준 라이브러리를 불러옵니다.
import os
import re
import urllib.parse
import urllib.request

# .env 파일에 저장된 비밀 키(예: OPENAI_API_KEY)를 로드하는 라이브러리를 불러옵니다.
from dotenv import load_dotenv

# LangChain에서 메모리 상에 대화 히스토리를 저장하고 관리하는 클래스를 불러옵니다.
from langchain_core.chat_history import InMemoryChatMessageHistory

# LangChain에서 프롬프트 템플릿 및 대화 기록 위치를 지정하는 모듈을 불러옵니다.
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# 대화 기록(History)을 자동으로 추적/연결해 주는 체인 실행기 객체를 불러옵니다.
from langchain_core.runnables.history import RunnableWithMessageHistory

# LangChain의 OpenAI 대화형 언어 모델(LLM) 연동 클래스를 불러옵니다.
from langchain_openai import ChatOpenAI

# Streamlit 웹 대화형 UI 프레임워크를 불러옵니다.
import streamlit as st

# .env 파일에서 환경변수(OPENAI_API_KEY)를 읽어와 시스템 환경변수로 설정합니다.
load_dotenv()

# ---------------------------------------------------------
# 1. Streamlit UI 웹페이지 및 세션 상태 초기화
# ---------------------------------------------------------

# 브라우저 탭의 제목, 아이콘, 페이지 레이아웃(wide: 화면 전체 넓이 사용)을 기본 설정합니다.
st.set_page_config(
    page_title="마음쉼터 - 감성 챗봇 & 뮤직 플레이어",
    page_icon="🎵",
    layout="wide",  # 사이드바와 메인 대화창 활용을 위해 wide 레이아웃 적용
)

# 세션 상태(st.session_state)에 대화 기록 저장소('store') 딕셔너리가 없으면 새로 생성합니다.
if "store" not in st.session_state:
    st.session_state.store = {}

# 세션 상태에 메시지별 추천 음악 데이터 저장소('music_data') 딕셔너리가 없으면 생성합니다.
if "music_data" not in st.session_state:
    st.session_state.music_data = {}

# 세션 상태에 현재 사이드바에서 재생 중인 음악 URL('current_playing_url') 변수가 없으면 None으로 초기화합니다.
if "current_playing_url" not in st.session_state:
    st.session_state.current_playing_url = None

# 세션 상태에 현재 사이드바에서 재생 중인 음악 제목('current_playing_title') 변수가 없으면 None으로 초기화합니다.
if "current_playing_title" not in st.session_state:
    st.session_state.current_playing_title = None


# 고유 세션 ID에 해당하는 대화 히스토리 객체를 반환하거나, 없으면 신규 생성하는 관리 함수입니다.
def get_session_history(session_id: str):
    # 전달받은 session_id가 저장소에 존재하지 않는 경우
    if session_id not in st.session_state.store:
        # 인메모리 대화 히스토리 객체를 생성합니다.
        history = InMemoryChatMessageHistory()
        # 첫 방문 사용자를 위한 AI 심리상담사의 기본 첫 인사 메시지를 추가합니다.
        history.add_ai_message(
            "안녕하세요. 오늘 하루는 어떠셨나요? ☕\n편안하게 당신의 마음을"
            " 나누어 주세요."
        )
        # 생성된 히스토리 객체를 세션 저장소 딕셔너리에 저장합니다.
        st.session_state.store[session_id] = history
    # 해당 세션 ID의 대화 히스토리 객체를 반환합니다.
    return st.session_state.store[session_id]


# ---------------------------------------------------------
# 2. 고속 YouTube 동영상 검색 함수
# ---------------------------------------------------------


# 입력된 검색어(query)로 유튜브 결과를 크롤링하여 동영상 URL 및 제목 정보를 수집하는 함수입니다.
def search_youtube_videos(query: str, max_results: int = 5) -> list[dict]:
    try:
        # 검색어 뒤에 " 음악"을 붙인 후 웹 URL 규칙에 맞게 퍼센트 인코딩(URL Encoding)합니다.
        search_query = urllib.parse.quote(query + " 음악")
        # 유튜브 검색 결과 페이지 요청 URL 문자열을 생성합니다.
        url = f"https://www.youtube.com/results?search_query={search_query}"

        # 유튜브 서버의 봇 차단을 방지하기 위해 일반 크롬 브라우저 헤더(User-Agent) 정보를 포함한 Request 객체를 생성합니다.
        req = urllib.request.Request(
            url,
            headers={
                "User-Agent": (
                    "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
                    " AppleWebKit/537.36 (KHTML, like Gecko)"
                    " Chrome/120.0.0.0 Safari/537.36"
                )
            },
        )

        # HTTP 요청을 보내고 응답 결과 HTML 페이지 전체 소스 코드를 UTF-8 텍스트로 변환하여 저장합니다.
        html = urllib.request.urlopen(req).read().decode("utf-8")
        # 정규표현식을 사용해 HTML 소스 코드 내에서 11자리 고유 비디오 ID 패턴('watch?v=비디오ID')을 모두 찾아 리스트로 반환합니다.
        video_ids = re.findall(r"watch\?v=([a-zA-Z0-9_-]{11})", html)

        # 파싱 결과를 담을 리스트와 중복 ID 검증용 세트(Set)를 선언합니다.
        results = []
        seen_ids = set()

        # 추출된 비디오 ID 목록을 순회 처리합니다.
        for vid in video_ids:
            # 이미 처리한 중복 비디오 ID가 아닌 경우에만 실행합니다.
            if vid not in seen_ids:
                # 중복 방지를 위해 집합(set)에 해당 비디오 ID를 등록합니다.
                seen_ids.add(vid)
                # 제목과 해당 유튜브 영상의 full URL 주소를 딕셔너리 형태로 리스트에 추가합니다.
                results.append({
                    "title": f"유튜브 음악 트랙 ({vid})",
                    "url": f"https://www.youtube.com/watch?v={vid}",
                })
                # 원하는 최대 수집 개수(max_results)에 도달하면 반복문을 탈출합니다.
                if len(results) >= max_results:
                    break

        # 완성된 수집 결과 딕셔너리 리스트를 반환합니다.
        return results
    except Exception:
        # 네트워크 오류 등 예외 발생 시 빈 리스트를 반환합니다.
        return []


# ---------------------------------------------------------
# 3. 사이드바 고정 음악 플레이어 영역
# ---------------------------------------------------------

# Streamlit 사이드바(왼쪽 고정 화면) UI 구역을 정의합니다.
with st.sidebar:
    # 사이드바 상단 헤더 타이틀을 표시합니다.
    st.header("🎧 마음쉼터 뮤직 오디오")
    # 플레이어에 대한 부연 안내 캡션 문구를 표시합니다.
    st.caption("대화 중에도 음악이 끊기지 않고 계속 재생됩니다.")

    # 세션 상태에 재생 등록된 음악 URL이 존재하는 경우 사이드바 오디오 플레이어를 화면에 출력합니다.
    if st.session_state.current_playing_url:
        # 재생 중인 트랙 제목을 초록색 강조 성공 박스(success callout)로 표시합니다.
        st.success(
            f"🎵 **재생 중:** {st.session_state.current_playing_title}"
        )
        # Streamlit 비디오/오디오 플레이어 컴포넌트를 통해 해당 유튜브 URL 영상을 재생합니다.
        st.video(st.session_state.current_playing_url)
    else:
        # 아직 선택된 음악이 없을 경우 안내 파란색 정보 박스(info callout)를 표시합니다.
        st.info("💡 대화창에서 위로의 추천 음악을 선택해 보세요!")

    # 구분선 스타일을 적용합니다.
    st.markdown("---")


# ---------------------------------------------------------
# 4. 음악 선택 목록 렌더링 함수
# ---------------------------------------------------------


# 메인 대화창에 추천된 음악 목록을 라디오 버튼 옵션으로 그려주는 렌더링 함수입니다.
def render_music_selector(music_list: list, msg_idx: int):
    # visual 구분선을 추가합니다.
    st.markdown("---")
    # 추천 음악 섹션의 서브헤더 제목을 표시합니다.
    st.subheader("🎵 추천 음악 리스트")

    # 라디오 버튼에 보여줄 '선택 라벨 문자열': '음악 딕셔너리 정보' 구조의 맵핑 딕셔너리를 작성합니다.
    options_dict = {
        f"트랙 #{i+1} : {item['url']}": item
        for i, item in enumerate(music_list)
    }

    # Streamlit 위젯 상태 충돌을 방지하기 위해 메시지 인덱스 기반의 고유한 key 이름을 지정합니다.
    radio_key = f"radio_{msg_idx}"

    # 화면에 트랙 선택 라디오 버튼 단일 선택 위젯을 생성하고 선택된 라벨 옵션을 받습니다.
    selected_label = st.radio(
        "듣고 싶은 트랙을 클릭하면 오른쪽 사이드바에서 연속 재생됩니다:",
        options=list(options_dict.keys()),
        key=radio_key,
    )

    # 사용자가 라디오 버튼 항목을 선택했을 때 실행되는 조건문입니다.
    if selected_label:
        # 선택한 라벨 문자열에 맵핑된 음악 정보 딕셔너리를 추출합니다.
        selected_item = options_dict[selected_label]
        # 현재 사이드바에서 재생 중인 URL과 새로 선택한 URL이 다른 경우에만 실행합니다.
        if (
            st.session_state.current_playing_url != selected_item["url"]
        ):
            # 세션 상태의 재생 URL을 새 음악 URL로 교체합니다.
            st.session_state.current_playing_url = selected_item["url"]
            # 세션 상태의 재생 제목을 새 트랙 라벨 문자열로 교체합니다.
            st.session_state.current_playing_title = selected_label
            # 사이드바 플레이어에 바뀐 음악을 즉시 반영하여 재생시키기 위해 페이지 전체를 즉시 재실행(Rerun)합니다.
            st.rerun()


# ---------------------------------------------------------
# 5. LangChain 감성 상담 챗봇 구성
# ---------------------------------------------------------

# AI 감성 상담사의 페르소나 및 응답 지침, 대화 맥락 플레이스홀더를 담은 프롬프트 템플릿을 정의합니다.
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """당신은 공감 능력이 뛰어난 따뜻한 감성 상담사입니다.
사용자의 이야기에서 기분이나 상태를 파악해 따뜻하게 위로와 공감을 전하세요.
답변 마지막 줄에는 반드시 사용자의 기분에 맞는 음악 검색 키워드를 아래 형식으로 작성해 주세요.

검색키워드: [기분/상태 관련 음악 검색어]
(예: 검색키워드: 그리울 때 듣는 잔잔한 노래)""",
        ),
        # 대화 히스토리 목록이 주입될 위치 플레이스홀더 지정
        MessagesPlaceholder(variable_name="history"),
        # 사용자 신규 입력을 받기 위한 인간(human) 메시지 지정
        ("human", "{input}"),
    ]
)

# OpenAI의 gpt-4o-mini 모델을 인스턴스화합니다. (temperature=0.7로 창의적이고 따뜻한 응답 유도)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)
# 프롬프트 템플릿과 LLM 모델을 파이프라인(|) 기호로 결합하여 기본 체인을 작성합니다.
chain = prompt | llm

# 세션 히스토리 자동 관리 기능이 결합된 최종 Runnable 대화 체인 객체를 생성합니다.
bot_with_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)

# ---------------------------------------------------------
# 6. 메인 화면 대화 기록 및 UI 구성
# ---------------------------------------------------------

# 메인 화면에 대화형 애플리케이션의 타이틀 제목을 표시합니다.
st.title("🎵 마음쉼터: 감성 챗봇 & 음악 플레이어")
# 메인 서브 안내 문구를 캡션 형태로 표시합니다.
st.caption(
    "오늘 어떤 기분이신가요? 마음을 이야기해 주시면 따뜻한 위로와 음악을"
    " 추천해 드립니다."
)

# 세션 상태에서 감성 상담 대화용 'emotional_music_session' 히스토리 객체를 가져옵니다.
session_history = get_session_history("emotional_music_session")

# 저장된 대화 기록(messages)을 순회하면서 이전 대화 내역들을 순서대로 화면에 출력합니다.
for idx, msg in enumerate(session_history.messages):
    # 메시지 타입이 'ai'이면 assistant, 아니면 user 역할을 지정합니다.
    role = "assistant" if msg.type == "ai" else "user"

    # 챗봇 메시지 스타일 컨테이너(chat_message)를 생성합니다.
    with st.chat_message(role):
        # AI 메시지의 경우 프롬프트 지침용 '검색키워드:' 문구를 분리하여 위로 텍스트 본문만 화면에 출력합니다.
        content_to_show = msg.content.split("검색키워드:")[0].strip()
        st.write(content_to_show)

        # AI 상담사의 메시지이면서 세션 음악 데이터에 해당 메시지 인덱스의 음악이 존재하는 경우
        if role == "assistant" and idx in st.session_state.music_data:
            # 세션에 저장되어 있던 추천 음악 트랙 목록 데이터를 가져옵니다.
            music_list = st.session_state.music_data[idx]
            # 음악 데이터 리스트가 비어있지 않은 경우 라디오 선택 플레이어 UI를 출력합니다.
            if music_list:
                render_music_selector(music_list, idx)

# ---------------------------------------------------------
# 7. 사용자 신규 입력 처리
# ---------------------------------------------------------

# 화면 하단 채팅 입력창(st.chat_input)에서 사용자의 입력값을 받으면 조건문 이하를 실행합니다.
if user_input := st.chat_input("당신의 이야기를 들려주세요..."):
    # 사용자가 입력한 메시지를 대화창 화면에 'user' 역할 스타일로 즉시 출력합니다.
    st.chat_message("user").write(user_input)

    # LangChain 대화 히스토리 구분을 위한 세션 식별 ID 세팅값을 딕셔너리로 준비합니다.
    config = {"configurable": {"session_id": "emotional_music_session"}}

    # 'assistant' 챗봇 역할 스타일 컨테이너 내에서 AI 응답 생성을 수행합니다.
    with st.chat_message("assistant"):
        # LangChain 대화 체인을 실행하여 사용자 입력에 대한 답변 객체를 생성 받아옵니다.
        response = bot_with_history.invoke({"input": user_input}, config=config)
        # LLM 응답 텍스트 전체 원문을 문자열 변수에 저장합니다.
        full_response = response.content

        search_keyword = ""
        display_text = full_response

        # AI 응답 내에 '검색키워드:' 구분 문구가 포함되어 포함된 경우 실행합니다.
        if "검색키워드:" in full_response:
            # '검색키워드:' 텍스트 기준으로 분할합니다.
            parts = full_response.split("검색키워드:")
            # 첫 번째 파트(상담 위로 문구)를 사용자 표시용 텍스트로 저장합니다.
            display_text = parts[0].strip()
            # 두 번째 파트(키워드)를 유튜브 음악 검색용 키워드로 지정합니다.
            search_keyword = parts[1].strip()
        else:
            # 구분 키워드가 없는 예외 상황의 경우 사용자 입력 문장에 "음악"을 붙여 기본 검색 키워드로 사용합니다.
            search_keyword = f"{user_input} 음악"

        # 파싱 완료된 AI 상담사의 위로 문구를 화면에 렌더링 출력합니다.
        st.write(display_text)

        # 현재 세션 메시지 히스토리에 방금 새로 추가된 AI 메시지의 인덱스 번호를 계산합니다.
        new_ai_index = len(session_history.messages) - 1

        # 유튜브 음악 검색 진행 동안 동작할 로딩 스피너(st.spinner)를 보여줍니다.
        with st.spinner(
            f"🎶 '{search_keyword}' 관련 음악을 검색 중입니다..."
        ):
            # 유튜브 동영상 실시간 고속 검색 함수를 호출하여 5개의 추천 음악 목록을 가져옵니다.
            music_list = search_youtube_videos(search_keyword)

        # 검색된 추천 음악 리스트를 세션의 메시지 인덱스 키값으로 보존 저장합니다.
        st.session_state.music_data[new_ai_index] = music_list

        # 수집된 검색 결과 음악 목록이 존재하는 경우 자동으로 첫 번째 트랙을 재생 등록합니다.
        if music_list:
            # 첫 번째 트랙의 URL을 세션 상태의 현재 재생 URL로 설정합니다.
            st.session_state.current_playing_url = music_list[0]["url"]
            # 첫 번째 트랙의 라벨 정보를 세션 상태의 현재 재생 제목으로 설정합니다.
            st.session_state.current_playing_title = (
                f"트랙 #1 : {music_list[0]['url']}"
            )
            # 메인 대화창 하단에 음악 선택 라디오 위젯 UI를 출력합니다.
            render_music_selector(music_list, new_ai_index)
            # 새로 등록된 음악을 사이드바 플레이어에 바로 적용 및 자동 재생하기 위해 앱을 재실행합니다.
            st.rerun()
        else:
            # 검색 결과가 존재하지 않을 때 안내 메시지를 출력합니다.
            st.info("관련 음악 영상 검색 결과를 불러오지 못했습니다.")

Writing app.py


In [3]:
%%writefile app.py

# ---------------------------------------------------------
# 0. 라이브러리 임포트 및 시스템 초기화
# ---------------------------------------------------------

# 파이썬 기본 경고 메시지(DeprecationWarning, UserWarning 등)가 콘솔에 출력되지 않도록 무시 처리합니다.
import warnings

warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# 운영체제 환경변수 접근, 정규표현식 파싱, URL 인코딩 및 HTTP 네트워크 통신 라이브러리를 불러옵니다.
import os
import re
import urllib.parse
import urllib.request

# .env 파일에 저장된 비밀 키(OPENAI_API_KEY)를 로드하기 위한 라이브러리를 불러옵니다.
from dotenv import load_dotenv

# LangChain에서 메모리 내 대화 히스토리를 저장하고 관리하는 클래스를 불러옵니다.
from langchain_core.chat_history import InMemoryChatMessageHistory

# LangChain 프롬프트 템플릿 및 대화 기록 위치를 지정하는 모듈을 불러옵니다.
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# 대화 기록(History)을 자동으로 추적/연결해 주는 체인 실행기 객체를 불러옵니다.
from langchain_core.runnables.history import RunnableWithMessageHistory

# LangChain의 OpenAI 대화형 언어 모델(LLM) 연동 클래스를 불러옵니다.
from langchain_openai import ChatOpenAI

# Streamlit 웹 대화형 UI 프레임워크를 불러옵니다.
import streamlit as st

# .env 파일에서 환경변수를 시스템 환경변수로 등록합니다.
load_dotenv()

# ---------------------------------------------------------
# 1. Streamlit UI 웹페이지 및 세션 상태 초기화
# ---------------------------------------------------------

# 브라우저 탭의 제목, 아이콘, 전체 넓이(wide) 레이아웃을 설정합니다.
st.set_page_config(
    page_title="마음쉼터 - 감성 챗봇 & 뮤직 플레이어",
    page_icon="🎵",
    layout="wide",
)

# 대화 기록 저장용 딕셔너리 세션 상태를 초기화합니다.
if "store" not in st.session_state:
    st.session_state.store = {}

# 메시지별 추천 음악 데이터 저장용 딕셔너리 세션 상태를 초기화합니다.
if "music_data" not in st.session_state:
    st.session_state.music_data = {}

# 현재 재생 중인 음악 URL 상태를 초기화합니다.
if "current_playing_url" not in st.session_state:
    st.session_state.current_playing_url = None

# 현재 재생 중인 음악 제목 상태를 초기화합니다.
if "current_playing_title" not in st.session_state:
    st.session_state.current_playing_title = None


# 고유 세션 ID에 해당하는 대화 히스토리 객체를 반환하거나 없으면 신규 생성합니다.
def get_session_history(session_id: str):
    if session_id not in st.session_state.store:
        history = InMemoryChatMessageHistory()
        # AI 상담사의 기본 첫 인사 메시지를 추가합니다.
        history.add_ai_message(
            "안녕하세요. 오늘 하루는 어떠셨나요? ☕\n편안하게 당신의 마음을"
            " 나누어 주세요."
        )
        st.session_state.store[session_id] = history
    return st.session_state.store[session_id]


# ---------------------------------------------------------
# 2. 고속 YouTube 동영상 검색 함수 (타임아웃 및 차단 방지 강화)
# ---------------------------------------------------------


# 검색어로 유튜브 동영상을 크롤링하여 비디오 URL 목록을 반환하는 함수입니다.
def search_youtube_videos(query: str, max_results: int = 5) -> list[dict]:
    try:
        # 검색어 뒤에 " 음악"을 덧붙인 뒤 URL 인코딩을 수행합니다.
        search_query = urllib.parse.quote(query + " 음악")
        url = f"https://www.youtube.com/results?search_query={search_query}"

        # 유튜브 봇 차단을 방지하기 위한 실제 사용자 브라우저 헤더 정보 세팅
        headers = {
            "User-Agent": (
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
                " AppleWebKit/537.36 (KHTML, like Gecko)"
                " Chrome/122.0.0.0 Safari/537.36"
            ),
            "Accept-Language": "ko-KR,ko;q=0.9,en-US;q=0.8,en;q=0.7",
        }

        req = urllib.request.Request(url, headers=headers)

        # network timeout을 10초로 지정하여 타임아웃 오류 발생 가능성을 방지합니다.
        with urllib.request.urlopen(req, timeout=10) as response:
            html = response.read().decode("utf-8")

        # 정규표현식을 통해 동영상 11자리 고유 비디오 ID 목록을 파싱합니다.
        video_ids = re.findall(r"/watch\?v=([a-zA-Z0-9_-]{11})", html)

        results = []
        seen_ids = set()

        for vid in video_ids:
            if vid not in seen_ids:
                seen_ids.add(vid)
                results.append({
                    "title": f"유튜브 음악 트랙 ({vid})",
                    "url": f"https://www.youtube.com/watch?v={vid}",
                })
                if len(results) >= max_results:
                    break

        return results
    except Exception:
        # 네트워크 연결 실패 또는 타임아웃 시 안전하게 빈 리스트를 반환합니다.
        return []


# ---------------------------------------------------------
# 3. 콜백 함수 및 라디오 버튼 렌더링
# ---------------------------------------------------------


# 트랙 라디오 버튼 클릭 이벤트(on_change) 발생 시 호출되는 콜백 함수입니다.
def on_music_select(radio_key: str, options_dict: dict):
    # 선택된 라벨 값을 세션 상태에서 가져옵니다.
    selected_label = st.session_state.get(radio_key)
    if selected_label and selected_label in options_dict:
        selected_item = options_dict[selected_label]
        # 사이드바 플레이어의 재생 URL 및 제목 정보를 즉시 갱신합니다.
        st.session_state.current_playing_url = selected_item["url"]
        st.session_state.current_playing_title = selected_label


# 메인 대화창 내 음악 추천 옵션 라디오 위젯 출력 함수입니다.
def render_music_selector(music_list: list, msg_idx: int):
    st.markdown("---")
    st.subheader("🎵 추천 음악 리스트")

    # 선택 옵션용 딕셔너리를 생성합니다.
    options_dict = {
        f"트랙 #{i+1} : {item['url']}": item
        for i, item in enumerate(music_list)
    }

    radio_key = f"radio_{msg_idx}"

    # 무한 루프 예방을 위해 on_change 콜백 방식을 적용하여 선택 상태 변화를 감지합니다.
    st.radio(
        "듣고 싶은 트랙을 클릭하면 오른쪽 사이드바에서 연속 재생됩니다:",
        options=list(options_dict.keys()),
        key=radio_key,
        on_change=on_music_select,
        args=(radio_key, options_dict),
    )


# ---------------------------------------------------------
# 4. 사이드바 고정 음악 플레이어 영역
# ---------------------------------------------------------

with st.sidebar:
    st.header("🎧 마음쉼터 뮤직 오디오")
    st.caption("대화 중에도 음악이 끊기지 않고 계속 재생됩니다.")

    # 세션에 재생 정보가 존재하면 오디오/비디오 플레이어를 출력합니다.
    if st.session_state.current_playing_url:
        st.success(
            f"🎵 **재생 중:** {st.session_state.current_playing_title}"
        )
        st.video(st.session_state.current_playing_url)
    else:
        st.info("💡 대화창에서 위로의 추천 음악을 선택해 보세요!")

    st.markdown("---")


# ---------------------------------------------------------
# 5. LangChain 감성 상담 챗봇 구성
# ---------------------------------------------------------

# 상담사 페르소나와 음악 키워드 추출을 유도하는 프롬프트 템플릿입니다.
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """당신은 공감 능력이 뛰어난 따뜻한 감성 상담사입니다.
사용자의 이야기에서 기분이나 상태를 파악해 따뜻하게 위로와 공감을 전하세요.
답변 마지막 줄에는 반드시 사용자의 기분에 맞는 음악 검색 키워드를 아래 형식으로 작성해 주세요.

검색키워드: [기분/상태 관련 음악 검색어]
(예: 검색키워드: 그리울 때 듣는 잔잔한 노래)""",
        ),
        MessagesPlaceholder(variable_name="history"),
        ("human", "{input}"),
    ]
)

# OpenAI gpt-4o-mini 모델을 연결합니다.
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)
chain = prompt | llm

# 대화 기록 자동 저장이 적용된 실행 체인을 생성합니다.
bot_with_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)


# ---------------------------------------------------------
# 6. 메인 화면 대화 기록 출력
# ---------------------------------------------------------

st.title("🎵 마음쉼터: 감성 챗봇 & 음악 플레이어")
st.caption(
    "오늘 어떤 기분이신가요? 마음을 이야기해 주시면 따뜻한 위로와 음악을"
    " 추천해 드립니다."
)

session_history = get_session_history("emotional_music_session")

# 저장되어 있는 이전 대화 히스토리를 화면에 순서대로 렌더링합니다.
for idx, msg in enumerate(session_history.messages):
    role = "assistant" if msg.type == "ai" else "user"

    with st.chat_message(role):
        # AI 답변 본문만 파싱하여 화면에 표시합니다.
        content_to_show = msg.content.split("검색키워드:")[0].strip()
        st.write(content_to_show)

        # 이전 답변에 연결된 음악 데이터가 있을 경우 추천 음악 라디오 위젯을 그립니다.
        if role == "assistant" and idx in st.session_state.music_data:
            music_list = st.session_state.music_data[idx]
            if music_list:
                render_music_selector(music_list, idx)


# ---------------------------------------------------------
# 7. 사용자 신규 입력 및 메시지 처리
# ---------------------------------------------------------

if user_input := st.chat_input("당신의 이야기를 들려주세요..."):
    # 입력한 메시지를 사용자의 말풍선으로 렌더링합니다.
    st.chat_message("user").write(user_input)

    config = {"configurable": {"session_id": "emotional_music_session"}}

    with st.chat_message("assistant"):
        # LLM 모델을 실행해 답변을 가져옵니다.
        response = bot_with_history.invoke({"input": user_input}, config=config)
        full_response = response.content

        search_keyword = ""
        display_text = full_response

        # 답변 본문과 검색 키워드를 분리합니다.
        if "검색키워드:" in full_response:
            parts = full_response.split("검색키워드:")
            display_text = parts[0].strip()
            search_keyword = parts[1].strip()
        else:
            search_keyword = f"{user_input} 음악"

        # 위로 텍스트 메시지를 화면에 출력합니다.
        st.write(display_text)

        new_ai_index = len(session_history.messages) - 1

        # 스피너 표시 동안 유튜브 크롤링 검색을 진행합니다.
        with st.spinner(
            f"🎶 '{search_keyword}' 관련 음악을 검색 중입니다..."
        ):
            music_list = search_youtube_videos(search_keyword)

        # 검색된 리스트를 세션 상태에 보존합니다.
        st.session_state.music_data[new_ai_index] = music_list

        if music_list:
            # 첫 번째 트랙을 기본 플레이어 재생 목록으로 세팅합니다.
            st.session_state.current_playing_url = music_list[0]["url"]
            st.session_state.current_playing_title = (
                f"트랙 #1 : {music_list[0]['url']}"
            )
            render_music_selector(music_list, new_ai_index)
            # 세션에 새로 추가된 첫 트랙을 사이드바 플레이어에 즉시 반영하기 위해 UI를 리런합니다.
            st.rerun()
        else:
            st.info("관련 음악 영상 검색 결과를 불러오지 못했습니다.")

Overwriting app.py


In [4]:
%%writefile app.py

# =========================================================
# 0. 경고 메시지 비활성화 및 라이브러리 임포트
# =========================================================

# 파이썬 실행 시 발생하는 불필요한 경고 메시지(DeprecationWarning, UserWarning)가
# 웹 UI 화면이나 터미널 콘솔에 출력되어 지저분해지는 것을 방지합니다.
import warnings

warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# JSON 데이터 파싱, 환경 변수 로드, 정규 표현식 검색, 웹 URL 요청을 위한 파이썬 표준 라이브러리들입니다.
import json
import os
import re
import urllib.parse
import urllib.request

# .env 파일에 저장된 OPENAI_API_KEY 등의 비밀 환경 변수를 읽어와 os.environ에 자동으로 등록합니다.
from dotenv import load_dotenv

# LangChain 최신 버전 표준: 인메모리(RAM) 기반 대화 이력 저장소 클래스입니다.
from langchain_core.chat_history import InMemoryChatMessageHistory

# LLM 프롬프트를 작성하고 대화 히스토리 위치를 지정하기 위한 템플릿 모듈입니다.
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# 사용자/세션 ID별로 대화 이력을 자동 추적하고 관리해주는 체인 래퍼(Wrapper)입니다.
from langchain_core.runnables.history import RunnableWithMessageHistory

# OpenAI의 LLM(대화형 인공지능) 모델을 연결하는 모듈입니다.
from langchain_openai import ChatOpenAI

# Python 전용 웹 애플리케이션 프레임워크인 Streamlit입니다.
import streamlit as st

# Streamlit 기본 요소 외에 커스텀 HTML/JavaScript(iframe 등)를 웹 화면에 직접 렌더링하기 위한 컴포넌트입니다.
import streamlit.components.v1 as components

# .env 파일에서 환경 변수를 실제로 불러오는 명령을 실행합니다.
load_dotenv()


# ---------------------------------------------------------
# 1. Streamlit UI 웹페이지 기본 설정 및 세션 상태(Session State) 초기화
# ---------------------------------------------------------
# 기능 설명:
# Streamlit은 사용자 입력이나 버튼 클릭 등 이벤트가 발생할 때마다 코드 전체를 위에서부터 아래로 재실행(Rerun)합니다.
# 따라서 변수값이 사라지지 않고 유지되려면 `st.session_state` 메모리 공간에 저장해야 합니다.

# 브라우저 탭의 제목, 파비콘 아이콘, 화면 배치 레이아웃(Wide: 넓은 화면 사용)을 설정합니다.
st.set_page_config(
    page_title="마음쉼터 - 감성 챗봇 & 뮤직 플레이어",
    page_icon="🎵",
    layout="wide",
)

# [세션 초기화 1] 사용자와 AI 간 전체 대화 히스토리 객체를 세션별로 보관할 딕셔너리 저장 공간입니다.
if "store" not in st.session_state:
    st.session_state.store = {}

# [세션 초기화 2] 챗봇 응답 메시지 인덱스(순서)별로 검색된 유튜브 음악 목록(URL, 썸네일, 제목 등)을 매핑해 보관합니다.
if "music_data" not in st.session_state:
    st.session_state.music_data = {}

# [세션 초기화 3] 현재 사이드바 오디오 플레이어에서 재생할/재생 중인 유튜브 11자리 비디오 ID를 저장합니다.
if "current_playing_id" not in st.session_state:
    st.session_state.current_playing_id = None

# [세션 초기화 4] 사이드바 플레이어 윗부분에 표시할 현재 재생 음악의 제목 텍스트를 저장합니다.
if "current_playing_title" not in st.session_state:
    st.session_state.current_playing_title = None

# [세션 초기화 5] 재생 버튼을 눌렀을 때 iframe 내에서 바로 소리가 나도록 자동재생(autoplay=1) 쿼리를 활성화할지 여부 제어 플래그입니다.
if "autoplay_flag" not in st.session_state:
    st.session_state.autoplay_flag = False


# 특정 세션 ID에 해당하는 대화 히스토리 객체를 가져오거나, 없으면 새로 생성하는 메모리 관리 함수입니다.
def get_session_history(session_id: str):
    # 전달받은 session_id 키가 st.session_state.store 딕셔너리에 없으면 새로 등록합니다.
    if session_id not in st.session_state.store:
        # 인메모리 대화 이력 객체를 생성합니다.
        history = InMemoryChatMessageHistory()
        # 최초 접속 시 사용자를 맞아줄 상담사의 안내 인사를 AI 메시지로 이력에 등록합니다.
        history.add_ai_message(
            "안녕하세요. 오늘 하루는 어떠셨나요? ☕\n편안하게 당신의 마음을 나누어 주세요."
        )
        # 해당 세션 ID의 이력 객체로 보관합니다.
        st.session_state.store[session_id] = history
    # 세션 ID에 대응되는 대화 히스토리 객체를 반환합니다.
    return st.session_state.store[session_id]


# ---------------------------------------------------------
# 2. 동적 수량 추출(정규식) 및 유튜브 크롤링/oEmbed 파싱 함수
# ---------------------------------------------------------
# 기능 설명:
# 사용자가 "신나는 음악 5곡 추천해줘" 또는 "슬픈 노래 10개"처럼 요청 수량을 정해 말할 때
# 정규표현식으로 숫자를 자동 감지하고, 유튜브 검색 결과 HTML 소스를 파싱하여 원하는 개수만큼 음악 정보를 가져옵니다.


# 문장(사용자 입력 또는 AI 검색어)에서 원하는 추천 음악 개수(숫자)를 추출하는 정규식 파싱 함수입니다.
def parse_max_results(text: str, default: int = 3) -> int:
    # 정규표현식: 숫자(\d+) 뒤에 '곡', '개', '트랙', '곳', '개수'와 같은 단어가 오는 패턴을 탐색합니다. (예: "5곡", "10개")
    match = re.search(r"(\d+)\s*(곡|개|트랙|곳|개수)", text)

    # 패턴이 감지된 경우 매칭된 숫자 그룹(group(1))을 정수로 변환하여 반환합니다.
    if match:
        count = int(match.group(1))
        # 시스템 안정성과 검색 속도를 위해 최소 1개, 최대 15개 범위로 제한 조절합니다.
        return min(max(count, 1), 15)

    # 문장에 수량 표현이 포함되어 있지 않으면 None을 반환하여 호출부에서 기본값(3곡)을 처리하도록 유도합니다.
    return None


# 유튜브 웹 페이지를 긁어와 비디오 ID를 추출하고 oEmbed API로 정확한 영상 제목을 파싱하는 함수입니다.
def search_youtube_videos(query: str, max_results: int = 3) -> list[dict]:
    try:
        # 검색어 문자열을 안전한 웹 URL 형태(퍼센트 인코딩, 예: 한글->%EB%A7%88%EC%9D%8C)로 변환합니다.
        search_query = urllib.parse.quote(query + " 음악")

        # 유튜브 검색 결과 URL을 생성합니다.
        url = f"https://www.youtube.com/results?search_query={search_query}"

        # 크롤링 차단(403 Forbidden)을 피하기 위해 일반 크롬 웹 브라우저가 보내는 요청 헤더(User-Agent)를 위장 설정합니다.
        req = urllib.request.Request(
            url,
            headers={
                "User-Agent": (
                    "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
                    " AppleWebKit/537.36 (KHTML, like Gecko)"
                    " Chrome/120.0.0.0 Safari/537.36"
                )
            },
        )

        # 유튜브 서버로 웹 요청을 보내 결과 HTML 페이지 원문 텍스트를 읽어옵니다.
        html = urllib.request.urlopen(req).read().decode("utf-8")

        # 정규표현식을 통해 HTML 코드 내에 숨겨진 유튜브 영상 고유 ID 11자리 문자열을 모두 추출합니다.
        video_ids = re.findall(r"watch\?v=([a-zA-Z0-9_-]{11})", html)

        results = []  # 파싱 결과를 담을 리스트
        seen_ids = set()  # 중복 추출 방지를 위한 고유 ID 저장 집합

        # 추출된 유튜브 비디오 ID 목록을 순회합니다.
        for vid in video_ids:
            # 이미 처리했던 비디오 ID가 아닌 경우에만 진행합니다.
            if vid not in seen_ids:
                seen_ids.add(vid)

                # 유튜브 영상 실제 접속 URL 및 고해상도/중해상도 썸네일 이미지 URL을 생성합니다.
                video_url = f"https://www.youtube.com/watch?v={vid}"
                thumbnail_url = (
                    f"https://img.youtube.com/vi/{vid}/mqdefault.jpg"
                )

                # 파싱 실패 시 기본으로 보여줄 대체 타이틀명을 지정해 둡니다.
                title = f"유튜브 음악 트랙 ({vid})"

                # 유튜브 공식 open-embed API(oEmbed)를 호출하여 가공되지 않은 해당 영상의 '진짜 원본 제목'을 가져옵니다.
                try:
                    oembed_url = f"https://www.youtube.com/oembed?url={video_url}&format=json"
                    oembed_req = urllib.request.Request(
                        oembed_url, headers={"User-Agent": "Mozilla/5.0"}
                    )
                    # 1.5초 이내에 응답이 올 경우에만 JSON 제목 데이터를 추출합니다.
                    with urllib.request.urlopen(
                        oembed_req, timeout=1.5
                    ) as response:
                        data = json.loads(response.read().decode("utf-8"))
                        title = data.get("title", title)
                except Exception:
                    # 응답 지연이나 타임아웃 발생 시 기본 타이틀을 유지하고 다음 로직을 진행합니다.
                    pass

                # 비디오 개별 데이터(ID, 제목, URL, 썸네일)를 딕셔너리로 묶어 결과 리스트에 추가합니다.
                results.append({
                    "id": vid,
                    "title": title,
                    "url": video_url,
                    "thumbnail": thumbnail_url,
                })

                # 사용자가 요구했던 최대 수량(max_results)만큼 수집이 완료되면 반복문을 탈출합니다.
                if len(results) >= max_results:
                    break

        return results
    except Exception:
        # 인터넷 연결 오류나 예외 상황 발생 시 프로그램이 멈추지 않도록 빈 리스트를 반환합니다.
        return []


# ---------------------------------------------------------
# 3. 사이드바 고정 음악 플레이어 영역 (iframe 자동 재생 기능)
# ---------------------------------------------------------
# 기능 설명:
# 좌측 사이드바에 유튜브 HTML5 embed 플레이어를 띄웁니다.
# 메인 대화 화면이 새로고침되거나 다른 대화를 나누더라도 사이드바 플레이어 영역은 별도로 유지되어 지속 재생이 가능합니다.

# 웹 앱 왼쪽 사이드바 영역 블록을 정의합니다.
with st.sidebar:
    st.header("🎧 마음쉼터 뮤직 오디오")
    st.caption("대화 중에도 음악이 끊기지 않고 계속 재생됩니다.")

    # 현재 선택된 재생 비디오 ID(current_playing_id)가 session_state에 등록되어 있는 경우 플레이어를 출력합니다.
    if st.session_state.current_playing_id:
        # 재생 중인 음악의 제목을 안내 상자로 표시합니다.
        st.success(
            f"🎵 **재생 중:**\n{st.session_state.current_playing_title}"
        )

        # 자동재생 플래그(autoplay_flag) 값에 따라 유튜브 URL 파라미터를 결정합니다. (1: 즉시 자동재생, 0: 대기)
        autoplay_param = "1" if st.session_state.autoplay_flag else "0"

        # iframe 내에서 바로 웹 오디오/비디오가 재생되도록 유튜브 임베드(embed) URL을 만듭니다.
        embed_url = f"https://www.youtube.com/embed/{st.session_state.current_playing_id}?autoplay={autoplay_param}&enablejsapi=1"

        # iframe 태그 HTML 문자열을 구성합니다. (allow 속성에 autoplay를 명시해야 브라우저에서 자동재생 허용)
        iframe_code = f"""
            <iframe 
                width="100%" 
                height="210" 
                src="{embed_url}" 
                title="YouTube video player" 
                frameborder="0" 
                allow="accelerometer; autoplay; clipboard-write; encrypted-media; gyroscope; picture-in-picture; web-share" 
                allowfullscreen>
            </iframe>
        """

        # Streamlit 사용자 정의 HTML 컴포넌트를 통해 웹 화면에 플레이어 iframe을 렌더링합니다.
        components.html(iframe_code, height=220)

        # 1회성 자동 재생 적용 후 다시 False로 원상 복구하여, 추후 단순 화면 재실행 시 음악이 원치 않게 계속 튀는 현상을 막습니다.
        st.session_state.autoplay_flag = False
    else:
        # 선택된 음악이 없을 때 출력되는 기본 안내 상자입니다.
        st.info("💡 대화창에서 위로의 추천 음악을 선택해 보세요!")

    # 구분선을 출력합니다.
    st.markdown("---")


# ---------------------------------------------------------
# 4. 카드형 음악 선택 목록 렌더링 함수 (UI Component & 자동 스크롤)
# ---------------------------------------------------------
# 기능 설명:
# AI 응답 메시지 하단에 추출된 음악 목록을 썸네일 이미지, 제목, [▶ 재생하기] 버튼 형태의 그리드 카드로 렌더링합니다.
# 렌더링 시 HTML Anchor 태그를 심어 자동으로 해당 음악 추천 카드 위치로 스크롤을 이동시킵니다.


def render_music_selector(music_list: list, msg_idx: int):
    st.markdown("---")
    
    # [추가] 스크롤 이동 대상이 될 고유 HTML ID 앵커 태그 생성
    st.markdown(f'<div id="music-section-{msg_idx}"></div>', unsafe_allow_html=True)
    st.subheader(f"🎵 추천 음악 리스트 ({len(music_list)}곡)")

    # 한 행(Row)에 최대 3개의 음악 카드가 가로로 배치되도록 컬럼 수(Column)를 계산합니다.
    num_cols = min(3, len(music_list))

    # 검색된 전체 음악 리스트를 num_cols(3개) 단위로 쪼개어 그리드 형태로 출력합니다.
    for i in range(0, len(music_list), num_cols):
        chunk = music_list[i : i + num_cols]
        # 컬럼 객체 생성 (예: 3개의 가로 칸 생성)
        cols = st.columns(len(chunk))

        for idx, item in enumerate(chunk):
            real_idx = i + idx  # 전체 음악 중 실제 순번(0, 1, 2, 3...)
            with cols[idx]:
                # 유튜브 썸네일 이미지를 카드 상단에 꽉 차게 렌더링합니다.
                st.image(item["thumbnail"], use_container_width=True)

                # 트랙 번호와 파싱된 유튜브 영상 제목을 굵은 글씨로 표시합니다.
                st.caption(f"**#{real_idx + 1}. {item['title']}**")

                # Streamlit 위젯 키 충돌 방지를 위해 메시지 순번(msg_idx)과 트랙 순번(real_idx)을 조합해 고유 Key를 만듭니다.
                btn_key = f"play_btn_{msg_idx}_{real_idx}"

                # 사용자가 '▶ 재생하기' 버튼을 누르면 해당 음악 정보가 세션에 저장되고 페이지가 재실행됩니다.
                if st.button(
                    "▶ 재생하기", key=btn_key, use_container_width=True
                ):
                    # 클릭한 동영상의 유튜브 ID를 세션에 저장하여 사이드바 플레이어와 연결합니다.
                    st.session_state.current_playing_id = item["id"]

                    # 클릭한 동영상의 제목을 세션에 저장합니다.
                    st.session_state.current_playing_title = (
                        f"#{real_idx + 1} {item['title']}"
                    )

                    # 버튼 클릭 즉시 음악이 플레이어에서 자동 재생되도록 플래그를 True로 바꿉니다.
                    st.session_state.autoplay_flag = True

                    # 전체 Streamlit 페이지를 다시 불러와 사이드바 오디오 플레이어를 즉시 갱신합니다.
                    st.rerun()

    # [추가] 해당 음악 목록 영역으로 브라우저 스크롤을 부드럽게 이동시키는 JavaScript 코드 실행
    js_scroll = f"""
    <script>
        setTimeout(function() {{
            var target = window.parent.document.getElementById("music-section-{msg_idx}");
            if (target) {{
                target.scrollIntoView({{ behavior: "smooth", block: "start" }});
            }}
        }}, 300);
    </script>
    """
    components.html(js_scroll, height=0)


# ---------------------------------------------------------
# 5. LangChain 기반 감성 상담 챗봇 구성 (System Prompt & Chain)
# ---------------------------------------------------------
# 기능 설명:
# AI에게 "따뜻한 심리 상담사" 페르소나를 부여하고,
# 답변 끝부분에 반드시 "검색키워드: [키워드]" 규격을 작성하도록 강제하는 프롬프트 템플릿입니다.

# 시스템 프롬프트 및 대화 히스토리 플레이스홀더를 정의합니다.
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """당신은 공감 능력이 뛰어난 따뜻한 감성 상담사입니다.
사용자의 이야기에서 기분이나 상태를 파악해 따뜻하게 위로와 공감을 전하세요.

답변 마지막 줄에는 반드시 사용자의 기분에 맞는 음악 검색 키워드를 아래 형식으로 작성해 주세요.

검색키워드: [기분/상태 관련 음악 검색어]
(예: 검색키워드: 신나고 기쁠 때 듣는 음악)""",
        ),
        # 대화 히스토리가 자동으로 주입될 변수 위치입니다.
        MessagesPlaceholder(variable_name="history"),
        # 사용자가 새로 입력한 텍스트가 전달되는 위치입니다.
        ("human", "{input}"),
    ]
)

# OpenAI의 경량화 고성능 모델인 'gpt-4o-mini'를 설정합니다. (temperature=0.7로 따뜻하고 창의적인 답변 유도)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

# 프롬프트와 LLM 모델을 파이프라인(|) 연산자로 연결해 기본 체인을 만듭니다.
chain = prompt | llm

# 체인에 대화 히스토리 관리자(RunnableWithMessageHistory)를 결합하여 사용자별 대화 맥락이 유지되도록 합니다.
bot_with_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",  # 사용자 입력 매개변수 키
    history_messages_key="history",  # 프롬프트 내 히스토리 위치 키
)


# ---------------------------------------------------------
# 6. 메인 대화 화면 및 이전 대화 기록 UI 렌더링
# ---------------------------------------------------------

# 메인 화면의 큰 타이틀과 설명 문구를 렌더링합니다.
st.title("🎵 마음쉼터: 감성 챗봇 & 음악 플레이어")
st.caption(
    "오늘 어떤 기분이신가요? 마음을 이야기해 주시면 따뜻한 위로와 음악을"
    " 추천해 드립니다."
)

# 지정된 식별자 키("emotional_music_session")를 사용하여 세션 대화 히스토리 객체를 가져옵니다.
session_history = get_session_history("emotional_music_session")

# 히스토리에 기록된 이전 모든 대화 메시지를 순서대로 화면에 재생성(렌더링)합니다.
for idx, msg in enumerate(session_history.messages):
    # 메시지 타입이 'ai'이면 assistant 말풍선, 아니면 user 말풍선을 지정합니다.
    role = "assistant" if msg.type == "ai" else "user"

    with st.chat_message(role):
        # AI 메시지의 경우 내부용 문구인 "검색키워드:" 부분은 잘라내어 사용자에게 깔끔한 위로 본문만 보여줍니다.
        content_to_show = msg.content.split("검색키워드:")[0].strip()
        st.write(content_to_show)

        # AI 상담사 응답에 해당하는 메시지이고, 해당 메시지 순번(idx)에 보관된 음악 데이터가 있다면 추천 음악 카드를 렌더링합니다.
        if role == "assistant" and idx in st.session_state.music_data:
            music_list = st.session_state.music_data[idx]
            if music_list:
                render_music_selector(music_list, idx)


# ---------------------------------------------------------
# 7. 신규 사용자 입력 처리 및 음악 검색 연동 (메인 이벤트 루프)
# ---------------------------------------------------------
# 기능 설명:
# 하단 입력창(chat_input)에 텍스트를 입력하면 다음과 같은 순서로 실행됩니다:
# 1) 입력값을 UI에 출력
# 2) AI 상담 응답 생성 및 본문/검색키워드 분리
# 3) 수량 파싱 (우선순위 1~3단계 적용)
# 4) 유튜브 자동 검색 실행 및 첫 곡 플레이어 자동 등록
# 5) st.rerun()을 통한 플레이어 즉시 갱신 및 화면 스크롤 이동

# 사용자가 대화 입력창에 메시지를 작성하고 엔터를 눌렀을 때 실행됩니다.
if user_input := st.chat_input("당신의 이야기를 들려주세요..."):
    # 사용자가 입력한 메시지를 화면 상에 유저 말풍선("user")으로 출력합니다.
    st.chat_message("user").write(user_input)

    # LangChain Runnable에 전달할 세션 설정 옵션입니다.
    config = {"configurable": {"session_id": "emotional_music_session"}}

    # AI 상담사의 답변 말풍선 영역 블록을 작성합니다.
    with st.chat_message("assistant"):
        # LangChain 대화 체인을 실행하여 AI의 답변을 받아옵니다.
        response = bot_with_history.invoke(
            {"input": user_input}, config=config
        )
        full_response = response.content

        search_keyword = ""
        display_text = full_response

        # AI의 응답 원문에서 순수 위로 텍스트 메시지와 시스템용 음악 검색 키워드를 분리 파싱합니다.
        if "검색키워드:" in full_response:
            parts = full_response.split("검색키워드:")
            display_text = parts[0].strip()  # 화면에 띄울 위로 본문
            search_keyword = parts[1].strip()  # 유튜브 검색에 사용할 키워드
        else:
            # 혹시 키워드 형식이 누락되었을 경우 사용자 입력을 검색어로 대체하는 예외 안전 처리입니다.
            search_keyword = f"{user_input} 음악"

        # 화면에 파싱된 깔끔한 AI 위로 텍스트만 출력합니다.
        st.write(display_text)

        # ---------------------------------------------------------
        # [동적 수량 결정 우선순위 로직]
        # ---------------------------------------------------------
        # 1순위: 사용자가 직접 입력창에 작성한 수량("5곡 추천해줘" -> 5)
        requested_count = parse_max_results(user_input)

        # 2순위: 사용자 입력에 숫자가 없으면 AI가 생성한 키워드에서 파싱
        if requested_count is None:
            requested_count = parse_max_results(search_keyword)

        # 3순위: 양쪽 모두 숫자가 지정되지 않은 경우 표준 기본값 3곡 적용
        if requested_count is None:
            requested_count = 3

        # 이번 대화로 세션 히스토리에 새롭게 추가된 AI 메시지의 인덱스 번호를 구합니다.
        new_ai_index = len(session_history.messages) - 1

        # 유튜브 검색을 진행하는 동안 사용자 화면에 로딩 애니메이션 스피너를 보여줍니다.
        with st.spinner(
            f"🎶 '{search_keyword}' 관련 음악 {requested_count}곡을 검색"
            " 중입니다..."
        ):
            # 설정된 검색어와 동적 수량(requested_count)으로 유튜브 크롤링 검색을 실행합니다.
            music_list = search_youtube_videos(
                query=search_keyword, max_results=requested_count
            )

        # 가져온 음악 트랙 목록을 세션 상태 딕셔너리에 이력 순번(new_ai_index) 키로 저장합니다.
        st.session_state.music_data[new_ai_index] = music_list

        # 검색된 음악 결과가 최소 1곡 이상 존재할 경우 실행합니다.
        if music_list:
            # 추천된 음악 목록의 첫 번째 트랙(#1)을 플레이어에 기본 재생곡으로 즉시 등록합니다.
            st.session_state.current_playing_id = music_list[0]["id"]
            st.session_state.current_playing_title = (
                f"#1 {music_list[0]['title']}"
            )

            # 최초 답변 시에도 플레이어에서 즉시 소리가 나며 자동 재생되도록 활성화 플래그를 True로 바꿉니다.
            st.session_state.autoplay_flag = True

            # 대화 말풍선 하단에 썸네일 카드형 선택 목록을 렌더링하고 자동 스크롤을 수행합니다.
            render_music_selector(music_list, new_ai_index)

            # 사이드바 플레이어의 비디오 ID 갱신 및 자동재생 반영을 위해 전체 Streamlit 페이지를 재실행(Rerun)합니다.
            st.rerun()
        else:
            # 검색 결과가 없을 시 사용자에게 안내 메시지를 띄웁니다.
            st.info("관련 음악 영상 검색 결과를 불러오지 못했습니다.")

Overwriting app.py
